# Import Modules

In [26]:
pip install streamlit

  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached pydeck-0.9.2-py2.py3-none-any.whl.metadata (4.2 kB)
  Using cached toml-0.10.2-py2.py3-none-any.whl.metadata (7.1 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 29.2 MB/s  0:00:00m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.6/797.6 kB 16.6 MB/s  0:00:00
Using cached blinker-1.9.0-py3-none-any.whl (8.5 kB)
Using cached pydeck-0.9.2-py2.py3-none-any.whl (11.3 MB)
Using cached toml-0.10.2-py2.py3-none-any.whl (16 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13/13 [streamlit]13 [streamlit]tipart]
Note: you may need to restart the kernel to use updated packages.


In [1]:
import sys
print(sys.executable)

/Users/siddharthravindran/venvs/emotion-classifier/bin/python


In [27]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import (f1_score, accuracy_score, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report,
                             recall_score, precision_score,
                             multilabel_confusion_matrix)
from sklearn.preprocessing import MultiLabelBinarizer
import matplotlib.pyplot as plt
from pylab import axes
import itertools
import os
import gc
import collections
import streamlit as st


import datasets
from datasets import Dataset, DatasetDict
import huggingface_hub
from huggingface_hub import login
datasets.disable_progress_bar()
import logging

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.nn import BCEWithLogitsLoss

import transformers
from transformers import (AutoTokenizer,
                          AutoModelForSequenceClassification,
                          AutoConfig, Adafactor,
                          Trainer, TrainingArguments,
                          EarlyStoppingCallback, TextClassificationPipeline,
                          pipeline)

from transformers.data.data_collator import DataCollatorWithPadding
from transformers import get_linear_schedule_with_warmup

from tqdm import tqdm

# import pytorch_lightning as pl
# from pytorch_lightning import (
#     Callback,
#     LightningDataModule,
#     LightningModule,
#     Trainer,
#     seed_everything,
# )
# from pytorch_lightning.loggers import Logger
# # from lightning_transformers.task.nlp.text_classification import (
# #     TextClassificationDataModule,
# #     TextClassificationTransformer,
# # )

import wandb


# from google.colab import drive, files
# drive.mount('/content/drive')

# %cd /content/drive/My\ Drive

#import necessary pre-processing functions
import sentiment_analysis_functions as saf
import importlib
importlib.reload(saf)

os.environ["TOKENIZERS_PARALLELISM"] = "false"

use_cuda = torch.cuda.is_available()
device = torch.device("cuda:0" if use_cuda else "cpu")

# Import Data

In [3]:
empathies = pd.read_csv("empathies.csv")
data = pd.read_csv("labeled_messages.csv")

# saf.save_data_to_HF(data, empathies)

In [4]:
data, id2label, label2id = saf.pre_process(data, empathies)
messages = saf.create_dataset_dict(data)



# Train Model

In [8]:
import torch
import numpy as np
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)
from sklearn.metrics import f1_score, precision_score, recall_score

# ----- config -----
MODEL = "roberta-base"
NUM_LABELS = len(label2id)
tokenizer = AutoTokenizer.from_pretrained(MODEL)

# messages should already exist from: messages = saf.create_dataset_dict(data)

# ----- tokenize + format labels -----
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=64)

def format_labels(batch):
    batch["labels"] = [[float(x) for x in lab] for lab in batch["labels"]]
    return batch

ds = messages.map(format_labels, batched=True).map(tokenize, batched=True)
ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

# ----- custom collator: forces float labels (required for multi-label BCE loss) -----
def collate_fn(features):
    return {
        "input_ids":      torch.stack([f["input_ids"] for f in features]),
        "attention_mask": torch.stack([f["attention_mask"] for f in features]),
        "labels":         torch.stack([f["labels"] for f in features]).float(),
    }

# ----- model -----
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",   # uses BCEWithLogitsLoss
    id2label=id2label,
    label2id=label2id,
)

# ----- metrics -----
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.sigmoid(torch.tensor(logits)).numpy()
    preds = (probs >= 0.5).astype(int)
    return {
        "f1_micro":  f1_score(labels, preds, average="micro", zero_division=0),
        "f1_macro":  f1_score(labels, preds, average="macro", zero_division=0),
        "precision": precision_score(labels, preds, average="micro", zero_division=0),
        "recall":    recall_score(labels, preds, average="micro", zero_division=0),
    }

# ----- training config -----
args = TrainingArguments(
    output_dir="emotion-classifier",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_micro",
    logging_steps=20,
    report_to="none",
)

# ----- train -----
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    compute_metrics=compute_metrics,
    data_collator=collate_fn,
)

trainer.train()
print(trainer.evaluate(ds["test"]))

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  0%|          | 0/705 [00:00<?, ?it/s]

{'loss': 0.586, 'grad_norm': 0.8159192800521851, 'learning_rate': 1.9432624113475178e-05, 'epoch': 0.14}
{'loss': 0.397, 'grad_norm': 0.5953819155693054, 'learning_rate': 1.8865248226950357e-05, 'epoch': 0.28}
{'loss': 0.3039, 'grad_norm': 0.4866793751716614, 'learning_rate': 1.8297872340425533e-05, 'epoch': 0.42}
{'loss': 0.2441, 'grad_norm': 0.4032047390937805, 'learning_rate': 1.773049645390071e-05, 'epoch': 0.57}
{'loss': 0.2026, 'grad_norm': 0.33986330032348633, 'learning_rate': 1.716312056737589e-05, 'epoch': 0.71}
{'loss': 0.1752, 'grad_norm': 0.2914091944694519, 'learning_rate': 1.6595744680851064e-05, 'epoch': 0.85}
{'loss': 0.1589, 'grad_norm': 0.27444347739219666, 'learning_rate': 1.6028368794326244e-05, 'epoch': 0.99}


  0%|          | 0/81 [00:00<?, ?it/s]

{'eval_loss': 0.13790681958198547, 'eval_f1_micro': 0.0, 'eval_f1_macro': 0.0, 'eval_precision': 0.0, 'eval_recall': 0.0, 'eval_runtime': 14.6227, 'eval_samples_per_second': 43.904, 'eval_steps_per_second': 5.539, 'epoch': 1.0}
{'loss': 0.1449, 'grad_norm': 0.24142315983772278, 'learning_rate': 1.546099290780142e-05, 'epoch': 1.13}
{'loss': 0.1356, 'grad_norm': 0.21247006952762604, 'learning_rate': 1.4893617021276596e-05, 'epoch': 1.27}
{'loss': 0.1296, 'grad_norm': 0.19512422382831573, 'learning_rate': 1.4326241134751775e-05, 'epoch': 1.41}
{'loss': 0.1263, 'grad_norm': 0.1695515215396881, 'learning_rate': 1.3758865248226951e-05, 'epoch': 1.55}
{'loss': 0.1247, 'grad_norm': 0.17214155197143555, 'learning_rate': 1.3191489361702127e-05, 'epoch': 1.7}
{'loss': 0.1189, 'grad_norm': 0.14896658062934875, 'learning_rate': 1.2624113475177307e-05, 'epoch': 1.84}
{'loss': 0.1154, 'grad_norm': 0.14831891655921936, 'learning_rate': 1.2056737588652483e-05, 'epoch': 1.98}


...

{'eval_loss': 0.09526453912258148, 'eval_f1_micro': 0.0, 'eval_f1_macro': 0.0, 'eval_precision': 0.0, 'eval_recall': 0.0, 'eval_runtime': 11.2545, 'eval_samples_per_second': 57.044, 'eval_steps_per_second': 7.197, 'epoch': 4.0}
{'loss': 0.108, 'grad_norm': 0.13354109227657318, 'learning_rate': 3.5460992907801423e-06, 'epoch': 4.1}
{'loss': 0.1083, 'grad_norm': 0.14968746900558472, 'learning_rate': 2.978723404255319e-06, 'epoch': 4.24}
{'loss': 0.1059, 'grad_norm': 0.1238127201795578, 'learning_rate': 2.4113475177304965e-06, 'epoch': 4.38}
{'loss': 0.1082, 'grad_norm': 0.17371869087219238, 'learning_rate': 1.8439716312056737e-06, 'epoch': 4.52}
{'loss': 0.1044, 'grad_norm': 0.1283198744058609, 'learning_rate': 1.276595744680851e-06, 'epoch': 4.66}
{'loss': 0.1062, 'grad_norm': 0.14124035835266113, 'learning_rate': 7.092198581560285e-07, 'epoch': 4.81}
{'loss': 0.107, 'grad_norm': 0.14104598760604858, 'learning_rate': 1.4184397163120568e-07, 'epoch': 4.95}


  0%|          | 0/81 [00:00<?, ?it/s]

{'eval_loss': 0.09473937004804611, 'eval_f1_micro': 0.0, 'eval_f1_macro': 0.0, 'eval_precision': 0.0, 'eval_recall': 0.0, 'eval_runtime': 11.8763, 'eval_samples_per_second': 54.057, 'eval_steps_per_second': 6.82, 'epoch': 4.98}
{'train_runtime': 1575.4541, 'train_samples_per_second': 7.173, 'train_steps_per_second': 0.447, 'train_loss': 0.1496525671465177, 'epoch': 4.98}


  0%|          | 0/81 [00:00<?, ?it/s]

{'eval_loss': 0.13760925829410553, 'eval_f1_micro': 0.0, 'eval_f1_macro': 0.0, 'eval_precision': 0.0, 'eval_recall': 0.0, 'eval_runtime': 16.8016, 'eval_samples_per_second': 38.27, 'eval_steps_per_second': 4.821, 'epoch': 4.982332155477032}


In [ ]:
# ----- model -----
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",   # uses BCEWithLogitsLoss
    id2label=id2label,
    label2id=label2id,
)

# compute pos_weight: how much to upweight positives per label
# (ratio of negatives to positives for each label, from training data)
train_labels = np.array([ex["labels"].tolist() for ex in ds["train"]])
pos_counts = train_labels.sum(axis=0)                    # positives per label
neg_counts = len(train_labels) - pos_counts              # negatives per label
pos_weight = torch.tensor(neg_counts / np.clip(pos_counts, 1, None), dtype=torch.float32)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = BCEWithLogitsLoss(pos_weight=pos_weight.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    compute_metrics=compute_metrics,
    data_collator=collate_fn,
)
trainer.train()

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  0%|          | 0/705 [00:00<?, ?it/s]

{'loss': 1.3383, 'grad_norm': 0.8273427486419678, 'learning_rate': 1.9432624113475178e-05, 'epoch': 0.14}
{'loss': 1.3391, 'grad_norm': 1.5843913555145264, 'learning_rate': 1.8865248226950357e-05, 'epoch': 0.28}
{'loss': 1.451, 'grad_norm': 1.4498385190963745, 'learning_rate': 1.8297872340425533e-05, 'epoch': 0.42}
{'loss': 1.373, 'grad_norm': 16.232444763183594, 'learning_rate': 1.773049645390071e-05, 'epoch': 0.57}
{'loss': 1.301, 'grad_norm': 1.297419548034668, 'learning_rate': 1.716312056737589e-05, 'epoch': 0.71}
{'loss': 1.3556, 'grad_norm': 9.549775123596191, 'learning_rate': 1.6595744680851064e-05, 'epoch': 0.85}
{'loss': 1.3971, 'grad_norm': 5.21035623550415, 'learning_rate': 1.6028368794326244e-05, 'epoch': 0.99}


  0%|          | 0/81 [00:00<?, ?it/s]

{'eval_loss': 1.1990599632263184, 'eval_f1_micro': 0.07564224548049477, 'eval_f1_macro': 0.08173108723495315, 'eval_precision': 0.039668184369737416, 'eval_recall': 0.8122605363984674, 'eval_runtime': 14.2489, 'eval_samples_per_second': 45.056, 'eval_steps_per_second': 5.685, 'epoch': 1.0}
{'loss': 1.2276, 'grad_norm': 10.35054874420166, 'learning_rate': 1.546099290780142e-05, 'epoch': 1.13}
{'loss': 1.2485, 'grad_norm': 3.213020086288452, 'learning_rate': 1.4893617021276596e-05, 'epoch': 1.27}
{'loss': 1.1305, 'grad_norm': 3.9454853534698486, 'learning_rate': 1.4326241134751775e-05, 'epoch': 1.41}
{'loss': 1.2128, 'grad_norm': 3.068010091781616, 'learning_rate': 1.3758865248226951e-05, 'epoch': 1.55}
{'loss': 1.2629, 'grad_norm': 3.4224960803985596, 'learning_rate': 1.3191489361702127e-05, 'epoch': 1.7}
{'loss': 1.1807, 'grad_norm': 4.174635410308838, 'learning_rate': 1.2624113475177307e-05, 'epoch': 1.84}
{'loss': 1.1039, 'grad_norm': 3.6623706817626953, 'learning_rate': 1.2056737588

...

  0%|          | 0/81 [00:00<?, ?it/s]

{'eval_loss': 0.7786940336227417, 'eval_f1_micro': 0.1716209939619136, 'eval_f1_macro': 0.1566192863681722, 'eval_precision': 0.09439264273853622, 'eval_recall': 0.9438058748403576, 'eval_runtime': 8.524, 'eval_samples_per_second': 75.317, 'eval_steps_per_second': 9.503, 'epoch': 4.0}
{'loss': 0.825, 'grad_norm': 2.1911423206329346, 'learning_rate': 3.5460992907801423e-06, 'epoch': 4.1}
{'loss': 0.772, 'grad_norm': 3.484218120574951, 'learning_rate': 2.978723404255319e-06, 'epoch': 4.24}
{'loss': 0.8045, 'grad_norm': 2.9628944396972656, 'learning_rate': 2.4113475177304965e-06, 'epoch': 4.38}
{'loss': 0.8198, 'grad_norm': 3.379946708679199, 'learning_rate': 1.8439716312056737e-06, 'epoch': 4.52}
{'loss': 0.8308, 'grad_norm': 3.414759397506714, 'learning_rate': 1.276595744680851e-06, 'epoch': 4.66}
{'loss': 0.7811, 'grad_norm': 2.0863254070281982, 'learning_rate': 7.092198581560285e-07, 'epoch': 4.81}
{'loss': 0.7915, 'grad_norm': 4.273825645446777, 'learning_rate': 1.4184397163120568e-0

  0%|          | 0/81 [00:00<?, ?it/s]

{'eval_loss': 0.7597119808197021, 'eval_f1_micro': 0.1750059283851079, 'eval_f1_macro': 0.15492116560787214, 'eval_precision': 0.09645797934910469, 'eval_recall': 0.9425287356321839, 'eval_runtime': 7.9656, 'eval_samples_per_second': 80.596, 'eval_steps_per_second': 10.169, 'epoch': 4.98}
{'train_runtime': 916.2948, 'train_samples_per_second': 12.332, 'train_steps_per_second': 0.769, 'train_loss': 1.0407540077858783, 'epoch': 4.98}


TrainOutput(global_step=705, training_loss=1.0407540077858783, metrics={'train_runtime': 916.2948, 'train_samples_per_second': 12.332, 'train_steps_per_second': 0.769, 'total_flos': 370656612581376.0, 'train_loss': 1.0407540077858783, 'epoch': 4.982332155477032})

In [9]:
# get raw predictions on validation set
pred_output = trainer.predict(ds["validation"])
probs = torch.sigmoid(torch.tensor(pred_output.predictions)).numpy()
labels = pred_output.label_ids

# sweep thresholds, pick best micro-F1
best_t, best_f1 = 0.5, 0.0
for t in np.arange(0.1, 0.6, 0.05):
    preds = (probs >= t).astype(int)
    f1 = f1_score(labels, preds, average="micro", zero_division=0)
    print(f"threshold {t:.2f}: micro-F1 = {f1:.3f}")
    if f1 > best_f1:
        best_t, best_f1 = t, f1

print(f"\nbest threshold: {best_t:.2f} (micro-F1 {best_f1:.3f})")

# evaluate test set at the best threshold
test_out = trainer.predict(ds["test"])
test_probs = torch.sigmoid(torch.tensor(test_out.predictions)).numpy()
test_preds = (test_probs >= best_t).astype(int)
print("\n=== TEST SET (threshold {:.2f}) ===".format(best_t))
print("micro-F1:", f1_score(test_out.label_ids, test_preds, average="micro", zero_division=0))
print("macro-F1:", f1_score(test_out.label_ids, test_preds, average="macro", zero_division=0))
print("precision:", precision_score(test_out.label_ids, test_preds, average="micro", zero_division=0))
print("recall:", recall_score(test_out.label_ids, test_preds, average="micro", zero_division=0))

  0%|          | 0/81 [00:00<?, ?it/s]

threshold 0.10: micro-F1 = 0.058
threshold 0.15: micro-F1 = 0.000
threshold 0.20: micro-F1 = 0.000
threshold 0.25: micro-F1 = 0.000
threshold 0.30: micro-F1 = 0.000
threshold 0.35: micro-F1 = 0.000
threshold 0.40: micro-F1 = 0.000
threshold 0.45: micro-F1 = 0.000
threshold 0.50: micro-F1 = 0.000
threshold 0.55: micro-F1 = 0.000

best threshold: 0.10 (micro-F1 0.058)


  0%|          | 0/81 [00:00<?, ?it/s]


=== TEST SET (threshold 0.10) ===
micro-F1: 0.057466677308643944
macro-F1: 0.018022128862054322
precision: 0.030638297872340424
recall: 0.4621309370988447


In [10]:
# 1. Check label shape and alignment
print("NUM_LABELS:", NUM_LABELS)
print("label2id sample:", dict(list(label2id.items())[:5]))

# 2. Look at one training example's label vector
sample = ds["train"][0]
print("label vector length:", len(sample["labels"]))
print("label vector:", sample["labels"])
print("active labels (indices where 1):", [i for i,v in enumerate(sample["labels"]) if v==1])
print("text:", messages["train"][0]["text"][:100])

# 3. How many active labels per example on average?
import numpy as np
all_labels = np.array([ex["labels"].tolist() for ex in ds["train"]])
print("avg active labels per example:", all_labels.sum(axis=1).mean())
print("label vector width:", all_labels.shape)

NUM_LABELS: 61
label2id sample: {'affectionate': 0, 'angry': 1, 'annoyed': 2, 'anxious': 3, 'average': 4}
label vector length: 61
label vector: tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0])
active labels (indices where 1): [22, 54]
text: i feel full and tired
avg active labels per example: 1.418141592920354
label vector width: (2260, 61)


In [11]:
# predict on validation using the ALREADY-TRAINED trainer
pred = trainer.predict(ds["validation"])
probs = torch.sigmoid(torch.tensor(pred.predictions)).numpy()
labels = pred.label_ids

print("logits shape:", probs.shape, "labels shape:", labels.shape)
print("max prob:", probs.max(), "mean max-prob per row:", probs.max(axis=1).mean())

# sweep thresholds
for t in [0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5]:
    preds = (probs >= t).astype(int)
    f1mi = f1_score(labels, preds, average="micro", zero_division=0)
    f1ma = f1_score(labels, preds, average="macro", zero_division=0)
    prec = precision_score(labels, preds, average="micro", zero_division=0)
    rec  = recall_score(labels, preds, average="micro", zero_division=0)
    print(f"t={t:.2f}  microF1={f1mi:.3f}  macroF1={f1ma:.3f}  P={prec:.3f}  R={rec:.3f}")

  0%|          | 0/81 [00:00<?, ?it/s]

logits shape: (642, 61) labels shape: (642, 61)
max prob: 0.12314162 mean max-prob per row: 0.12165041
t=0.10  microF1=0.058  macroF1=0.018  P=0.031  R=0.464
t=0.15  microF1=0.000  macroF1=0.000  P=0.000  R=0.000
t=0.20  microF1=0.000  macroF1=0.000  P=0.000  R=0.000
t=0.25  microF1=0.000  macroF1=0.000  P=0.000  R=0.000
t=0.30  microF1=0.000  macroF1=0.000  P=0.000  R=0.000
t=0.35  microF1=0.000  macroF1=0.000  P=0.000  R=0.000
t=0.40  microF1=0.000  macroF1=0.000  P=0.000  R=0.000
t=0.45  microF1=0.000  macroF1=0.000  P=0.000  R=0.000
t=0.50  microF1=0.000  macroF1=0.000  P=0.000  R=0.000


In [15]:
# count positives per label across the full dataset
train_labels = np.array([ex["labels"].tolist() for ex in ds["train"]])
counts = train_labels.sum(axis=0)
id2label_sorted = sorted(label2id.items(), key=lambda x: x[1])
for label, idx in id2label_sorted:
    print(f"{label:15s}: {int(counts[idx])}")

affectionate   : 24
angry          : 87
annoyed        : 117
anxious        : 155
average        : 76
better         : 57
bored          : 84
calm           : 123
confidence     : 42
confused       : 82
creative       : 4
depressed      : 111
despair        : 21
determined     : 62
disconnected   : 74
distortion     : 44
embarrassed    : 45
emotional      : 24
emotionless    : 80
engaged        : 38
excited        : 99
flustered      : 17
full           : 21
good           : 40
great          : 32
happy          : 177
healthy        : 4
hopeful        : 14
hungry         : 40
idk            : 46
ignored        : 5
inspired       : 17
intoxicated    : 28
jealous        : 5
lazy           : 25
lonely         : 91
loss           : 25
loved          : 6
okay           : 101
pain           : 98
pensive        : 37
playful        : 2
regret         : 31
relief         : 26
restless       : 14
rude           : 2
sad            : 181
scared         : 41
sexy           : 9
sick           : 36
s

In [16]:
EMOTION_FAMILIES = {
    # JOY (happy 177 + excited 99 + good 40 + great 32 + better 57 + loved 6 + affectionate 24 + playful 2)
    "happy": "joy", "excited": "joy", "good": "joy", "great": "joy",
    "better": "joy", "loved": "joy", "affectionate": "joy", "playful": "joy",
    # SADNESS (sad 181 + depressed 111 + lonely 91 + despair 21 + loss 25 + vulnerable 22 + emotional 24)
    "sad": "sadness", "depressed": "sadness", "lonely": "sadness",
    "despair": "sadness", "loss": "sadness", "vulnerable": "sadness", "emotional": "sadness",
    # ANGER (annoyed 117 + angry 87 + upset 34 + jealous 5 + rude 2 + unsatisfied 48)
    "annoyed": "anger", "angry": "anger", "upset": "anger",
    "jealous": "anger", "rude": "anger", "unsatisfied": "anger",
    # ANXIETY/FEAR (anxious 155 + stress 84 + uneasy 39 + restless 14 + scared 41 + uncertain 20 + flustered 17 + embarrassed 45)
    "anxious": "anxiety", "stress": "anxiety", "uneasy": "anxiety",
    "restless": "anxiety", "scared": "anxiety", "uncertain": "anxiety",
    "flustered": "anxiety", "embarrassed": "anxiety",
    # CALM/CONTENT (calm 123 + okay 101 + average 76 + stable 31 + relief 26 + pensive 37)
    "calm": "calm", "okay": "calm", "average": "calm",
    "stable": "calm", "relief": "calm", "pensive": "calm",
    # FATIGUE/LOW-ENERGY (tired 166 + sleep 80 + emotionless 80 + bored 84 + lazy 25 + unmotivated 58 + disconnected 74)
    "tired": "fatigue", "sleep": "fatigue", "emotionless": "fatigue",
    "bored": "fatigue", "lazy": "fatigue", "unmotivated": "fatigue", "disconnected": "fatigue",
    # PHYSICAL (pain 98 + sick 36 + hungry 40 + full 21 + intoxicated 28 + healthy 4 + sexy 9)
    "pain": "physical", "sick": "physical", "hungry": "physical",
    "full": "physical", "intoxicated": "physical", "healthy": "physical", "sexy": "physical",
    # ENGAGED/POSITIVE-ACTIVATION (confidence 42 + determined 62 + engaged 38 + inspired 17 + hopeful 14 + creative 4)
    "confidence": "engaged", "determined": "engaged", "engaged": "engaged",
    "inspired": "engaged", "hopeful": "engaged", "creative": "engaged",
    # CONFUSED (confused 82 + distortion 44 + idk 46 + uncertain already in anxiety)
    "confused": "confused", "distortion": "confused", "idk": "confused",
    # drop: surprised 3, ignored 5, regret 31→? (put regret in sadness if you want)
    "regret": "sadness", "surprised": "joy", "ignored": "sadness",
}

In [17]:
def consolidate_to_families(data, families_map):
    # 'multiple_empathies' style: each row has a list of original emotion strings.
    # Rebuild from the empathy columns you already created (empathy_0..empathy_4).
    emp_cols = [c for c in data.columns if c.startswith("empathy_")]
    def row_families(row):
        fams = set()
        for c in emp_cols:
            val = str(row[c]).strip()
            if val and val in families_map:
                fams.add(families_map[val])
        return sorted(fams)
    data["families"] = data.apply(row_families, axis=1)

    mlb = MultiLabelBinarizer()
    Y = mlb.fit_transform(data["families"])
    fam_id2label = {i: l for i, l in enumerate(mlb.classes_)}
    fam_label2id = {l: i for i, l in enumerate(mlb.classes_)}
    data["labels_families_one_hot"] = list(Y)
    return data, fam_id2label, fam_label2id, mlb.classes_

data, id2label, label2id, fam_classes = consolidate_to_families(data, EMOTION_FAMILIES)
NUM_LABELS = len(label2id)
print("families:", list(fam_classes))
print("NUM_LABELS:", NUM_LABELS)
print("avg families per example:", np.mean([len(f) for f in data["families"]]))
# check counts per family
fam_counts = np.array(list(data["labels_families_one_hot"])).sum(axis=0)
for i, c in enumerate(fam_counts):
    print(f"{fam_classes[i]:12s}: {int(c)}")

families: ['anger', 'anxiety', 'calm', 'confused', 'engaged', 'fatigue', 'joy', 'physical', 'sadness']
NUM_LABELS: 9
avg families per example: 1.1713841368584759
anger       : 294
anxiety     : 477
calm        : 536
confused    : 228
engaged     : 190
fatigue     : 671
joy         : 574
physical    : 273
sadness     : 523


In [21]:
# ---- 1. pull text + 9-family one-hot labels from consolidated `data` ----
X = data["message"].astype(str).tolist()
y = np.array(list(data["labels_families_one_hot"]))   # shape (n, 9)

print("texts:", len(X), "label matrix:", y.shape)   # expect (n, 9)

# ---- 2. train / val / test split (80 / 10 / 10) ----
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

def make_ds(texts, labels):
    return Dataset.from_dict({
        "text": texts,
        "labels": [[float(v) for v in row] for row in labels],   # float for BCE
    })

messages = DatasetDict({
    "train":      make_ds(X_train, y_train),
    "validation": make_ds(X_val,   y_val),
    "test":       make_ds(X_test,  y_test),
})

# ---- 3. tokenize ----
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=64)

ds = messages.map(tokenize, batched=True)
ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

# ---- 4. verify ----
print("label vector length:", len(ds["train"][0]["labels"]))   # MUST be 9
print("sample labels:", ds["train"][0]["labels"])
print("train/val/test sizes:", len(ds["train"]), len(ds["validation"]), len(ds["test"]))

texts: 3215 label matrix: (3215, 9)
label vector length: 9
sample labels: tensor([0., 0., 0., 0., 0., 1., 0., 0., 1.])
train/val/test sizes: 2572 321 322


In [23]:
# fresh model with 9 outputs
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=NUM_LABELS,           # = 9 now
    problem_type="multi_label_classification",
    id2label=id2label,
    label2id=label2id,
)

def collate_fn(features):
    return {
        "input_ids":      torch.stack([f["input_ids"] for f in features]),
        "attention_mask": torch.stack([f["attention_mask"] for f in features]),
        "labels":         torch.stack([f["labels"] for f in features]).float(),
    }

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.sigmoid(torch.tensor(logits)).numpy()
    preds = (probs >= 0.5).astype(int)
    return {
        "f1_micro":  f1_score(labels, preds, average="micro", zero_division=0),
        "f1_macro":  f1_score(labels, preds, average="macro", zero_division=0),
        "precision": precision_score(labels, preds, average="micro", zero_division=0),
        "recall":    recall_score(labels, preds, average="micro", zero_division=0),
    }

args = TrainingArguments(
    output_dir="emotion-classifier-v2",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_micro",
    logging_steps=20,
    report_to="none",
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=ds["train"], eval_dataset=ds["validation"],
    compute_metrics=compute_metrics,
    data_collator=collate_fn,
)

trainer.train()
print(trainer.evaluate(ds["test"]))

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  0%|          | 0/805 [00:00<?, ?it/s]

{'loss': 0.5343, 'grad_norm': 0.839719295501709, 'learning_rate': 1.950310559006211e-05, 'epoch': 0.12}
{'loss': 0.3854, 'grad_norm': 0.7687708139419556, 'learning_rate': 1.9006211180124224e-05, 'epoch': 0.25}
{'loss': 0.3736, 'grad_norm': 0.7693644165992737, 'learning_rate': 1.8509316770186337e-05, 'epoch': 0.37}
{'loss': 0.3667, 'grad_norm': 0.8083560466766357, 'learning_rate': 1.801242236024845e-05, 'epoch': 0.5}
{'loss': 0.3397, 'grad_norm': 0.8806576132774353, 'learning_rate': 1.751552795031056e-05, 'epoch': 0.62}
{'loss': 0.3152, 'grad_norm': 1.0545161962509155, 'learning_rate': 1.7018633540372672e-05, 'epoch': 0.75}
{'loss': 0.2905, 'grad_norm': 3.3621859550476074, 'learning_rate': 1.6521739130434785e-05, 'epoch': 0.87}
{'loss': 0.2827, 'grad_norm': 1.1848020553588867, 'learning_rate': 1.6024844720496894e-05, 'epoch': 0.99}


  0%|          | 0/41 [00:00<?, ?it/s]

{'eval_loss': 0.24957071244716644, 'eval_f1_micro': 0.5631768953068592, 'eval_f1_macro': 0.4259603321157141, 'eval_precision': 0.8571428571428571, 'eval_recall': 0.41935483870967744, 'eval_runtime': 24.6286, 'eval_samples_per_second': 13.034, 'eval_steps_per_second': 1.665, 'epoch': 1.0}
{'loss': 0.2526, 'grad_norm': 1.4449691772460938, 'learning_rate': 1.5527950310559007e-05, 'epoch': 1.12}
{'loss': 0.2459, 'grad_norm': 1.6053383350372314, 'learning_rate': 1.5031055900621118e-05, 'epoch': 1.24}
{'loss': 0.2273, 'grad_norm': 1.2929608821868896, 'learning_rate': 1.4534161490683232e-05, 'epoch': 1.37}
{'loss': 0.2183, 'grad_norm': 1.6547725200653076, 'learning_rate': 1.4037267080745342e-05, 'epoch': 1.49}
{'loss': 0.2143, 'grad_norm': 2.5155954360961914, 'learning_rate': 1.3540372670807453e-05, 'epoch': 1.61}
{'loss': 0.193, 'grad_norm': 2.907811164855957, 'learning_rate': 1.3043478260869566e-05, 'epoch': 1.74}
{'loss': 0.1991, 'grad_norm': 2.715031147003174, 'learning_rate': 1.254658385

...

{'eval_loss': 0.14405333995819092, 'eval_f1_micro': 0.8379888268156425, 'eval_f1_macro': 0.8341986809286362, 'eval_precision': 0.872093023255814, 'eval_recall': 0.8064516129032258, 'eval_runtime': 6.6509, 'eval_samples_per_second': 48.264, 'eval_steps_per_second': 6.165, 'epoch': 4.0}
{'loss': 0.1089, 'grad_norm': 2.7832958698272705, 'learning_rate': 3.6024844720496897e-06, 'epoch': 4.1}
{'loss': 0.1158, 'grad_norm': 3.650602340698242, 'learning_rate': 3.1055900621118013e-06, 'epoch': 4.22}
{'loss': 0.1055, 'grad_norm': 1.9363871812820435, 'learning_rate': 2.6086956521739132e-06, 'epoch': 4.35}
{'loss': 0.1139, 'grad_norm': 1.8654054403305054, 'learning_rate': 2.111801242236025e-06, 'epoch': 4.47}
{'loss': 0.0983, 'grad_norm': 2.0738108158111572, 'learning_rate': 1.6149068322981367e-06, 'epoch': 4.6}
{'loss': 0.116, 'grad_norm': 2.4196560382843018, 'learning_rate': 1.1180124223602485e-06, 'epoch': 4.72}
{'loss': 0.1152, 'grad_norm': 3.8855433464050293, 'learning_rate': 6.21118012422360

  0%|          | 0/41 [00:00<?, ?it/s]

{'eval_loss': 0.14267323911190033, 'eval_f1_micro': 0.8421052631578947, 'eval_f1_macro': 0.8399534752721822, 'eval_precision': 0.8685714285714285, 'eval_recall': 0.8172043010752689, 'eval_runtime': 3.9665, 'eval_samples_per_second': 80.927, 'eval_steps_per_second': 10.337, 'epoch': 5.0}
{'train_runtime': 1429.8398, 'train_samples_per_second': 8.994, 'train_steps_per_second': 0.563, 'train_loss': 0.19301676787204625, 'epoch': 5.0}


  0%|          | 0/41 [00:00<?, ?it/s]

{'eval_loss': 0.14031875133514404, 'eval_f1_micro': 0.8456375838926175, 'eval_f1_macro': 0.824565116927967, 'eval_precision': 0.8823529411764706, 'eval_recall': 0.8118556701030928, 'eval_runtime': 4.6729, 'eval_samples_per_second': 68.908, 'eval_steps_per_second': 8.774, 'epoch': 5.0}




# Inference

In [24]:
def predict_emotions(text, threshold=0.4):
    model.eval()
    inputs = tokenizer(text, truncation=True, padding=True, max_length=64, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.sigmoid(logits)[0].cpu().numpy()
    # map each family to its probability
    results = {id2label[i]: float(probs[i]) for i in range(len(probs))}
    results = dict(sorted(results.items(), key=lambda x: x[1], reverse=True))
    predicted = [fam for fam, p in results.items() if p >= threshold]
    return predicted, results

# try it
preds, scores = predict_emotions("i feel exhausted and a little hopeless today")
print("predicted families:", preds)
for fam, p in scores.items():
    print(f"  {fam:10s}: {p:.2f}")

predicted families: ['sadness', 'fatigue']
  sadness   : 0.84
  fatigue   : 0.57
  anxiety   : 0.06
  confused  : 0.06
  anger     : 0.04
  physical  : 0.04
  engaged   : 0.03
  calm      : 0.03
  joy       : 0.02


In [28]:
model.save_pretrained("./emotion-model")
tokenizer.save_pretrained("./emotion-model")

('./emotion-model/tokenizer_config.json',
 './emotion-model/special_tokens_map.json',
 './emotion-model/vocab.json',
 './emotion-model/merges.txt',
 './emotion-model/added_tokens.json',
 './emotion-model/tokenizer.json')

In [ ]:
# app.py

st.set_page_config(page_title="Emotion Classifier", page_icon="🎭")

@st.cache_resource
def load_model():
    tok = AutoTokenizer.from_pretrained("./emotion-model")
    mdl = AutoModelForSequenceClassification.from_pretrained("./emotion-model")
    mdl.eval()
    return tok, mdl

tokenizer, model = load_model()
id2label = model.config.id2label

st.title("🎭 Emotion Classifier")
st.caption("Multi-label emotion detection across 9 families — fine-tuned RoBERTa")

text = st.text_area("Type how you're feeling:", "I'm exhausted but weirdly hopeful about tomorrow.")
threshold = st.slider("Confidence threshold", 0.1, 0.9, 0.4, 0.05)

if st.button("Analyze") and text.strip():
    inputs = tokenizer(text, truncation=True, max_length=64, return_tensors="pt")
    with torch.no_grad():
        probs = torch.sigmoid(model(**inputs).logits)[0].numpy()
    scored = sorted(
        [(id2label[i], float(probs[i])) for i in range(len(probs))],
        key=lambda x: x[1], reverse=True
    )
    predicted = [f for f, p in scored if p >= threshold]
    st.subheader("Detected: " + (", ".join(predicted) if predicted else "—"))
    for fam, p in scored:
        st.write(f"**{fam}**")
        st.progress(p)

In [ ]:
class EmotionDataModule(pl.LightningDataModule):
  def __init__(self, train_dict, val_dict, test_dict, tokenizer, batch_size=8, max_token_len=128):
    super().__init__()
    self.batch_size = batch_size
    self.train_dict = train_dict
    self.val_dict = val_dict
    self.test_dict = test_dict
    self.tokenizer = tokenize
    self.max_token_len = max_token_len
  def setup(self, stage=None):
    self.train_dataset = EmotionDataset(
      self.train_dict,
      self.tokenizer,
    )
    self.val_dataset = EmotionDataset(
      self.val_dict,
      self.tokenizer,
    )
    self.test_dataset = EmotionDataset(
      self.test_dict,
      self.tokenizer,
    )
  def train_dataloader(self):
    return DataLoader(
      self.train_dataset,
      batch_size=self.batch_size,
      shuffle=True,
      num_workers=2
    )
  def val_dataloader(self):
    return DataLoader(
      self.test_dataset,
      batch_size=self.batch_size,
      num_workers=2
    )
  def test_dataloader(self):
    return DataLoader(
      self.test_dataset,
      batch_size=self.batch_size,
      num_workers=2
    )

In [ ]:
data_module = EmotionDataModule(
  messages['train'],
  messages['validation'],
  messages['test'],
  tokenizer,
  batch_size=8,
  max_token_len=128
)

In [ ]:
gc.collect()

torch.cuda.empty_cache()

In [ ]:
!nvidia-smi

Wed Aug 31 17:27:45 2022       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 460.32.03    Driver Version: 460.32.03    CUDA Version: 11.2     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  Tesla P100-PCIE...  Off  | 00000000:00:04.0 Off |                    0 |
| N/A   35C    P0    26W / 250W |      0MiB / 16280MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

In [ ]:
# def show_gpu(msg):
#     """
#     ref: https://discuss.pytorch.org/t/access-gpu-memory-usage-in-pytorch/3192/4
#     """
#     def query(field):
#         return(subprocess.check_output(
#             ['nvidia-smi', f'--query-gpu={field}',
#                 '--format=csv,nounits,noheader'],
#             encoding='utf-8'))
#     def to_int(result):
#         return int(result.strip().split('\n')[0])

#     used = to_int(query('memory.used'))
#     total = to_int(query('memory.total'))
#     pct = used/total
#     print('\n' + msg, f'{100*pct:2.1f}% ({used} out of {total})')


In [ ]:
# num_of_empathies = [len(y) for y in data["empathy"].apply(lambda x: x.split())]

# data["number_of_empathies"] = num_of_empathies

In [ ]:
# empathies_mapping = dict(zip(data["empathy_0"].unique(), np.arange(0, 60)))
# data["label_0"] = data["empathy_0"].map(empathies_mapping)
# data["label_0"] = data["label_0"].astype("int")

# data.drop(data[data["empathy_0"]=="loved"].index, axis = 0, inplace = True)
# data.reset_index(inplace = True)
# data.drop(["index"], axis = 1, inplace = True)